# Computation of the hCF for the Hansen-Zuazua system

The system equations read

$$
\ddot x_1(z,t) = x_1^{''}(z,t), \quad
\ddot x_2(z,t) = x_2^{''}(z,t)
$$

with the boundary conditions

$$
x_1(0,t) = x_2(0,t) = \xi(t),\quad
M \ddot x_1(0,t) = -x_1'(0,t) + x_2'(0,t)
$$

as well as


$$
x_1'(-l_1,t) = -u_1(t),\quad
x_2'(l_2,t) = u_2(t).
$$


## Input parametrisation

In [1]:
import sympy as sp
import chimera as ch

# symbols
(l1, l2, mass, z, t, tau) = sp.symbols("l_1 l_2 M z t tau")
y1, y2, u1, u2 = [func(t) for func in sp.symbols("y_1 y_2 u_1 u_2", cls=sp.Function)]
# operators
d_sym, s_sym = sp.symbols("d_t sigma", positive=True)

# shorthands
a1 = b1 = d_sym**2 + 2*d_sym
a2 = b2 = d_sym**2 - 2*d_sym
u_vec = sp.Matrix([u1, u2])
y_vec = sp.Matrix([y1, y2])

# general IO matrix
g_mat = sp.Rational(1,4) * sp.Matrix([
    [a1 * s_sym**l1 + a2 * s_sym**(-l1), -s_sym**l1 -s_sym**(-l1)], 
    [b1 * s_sym**l2 + b2 * s_sym**(-l2), s_sym**l2 +s_sym**(-l2)]])

# substitute actual values
g_mat = g_mat.subs([(l1, sp.pi), (l2, 10), (mass, 1)])

print("Parametrisation of the system input by the flat output:")
display(sp.Eq(u_vec, sp.MatMul(g_mat, y_vec)))

Parametrisation of the system input by the flat output:


Eq(Matrix([
[u_1(t)],
[u_2(t)]]), Matrix([
[sigma**pi*(d_t**2 + 2*d_t)/4 + (d_t**2 - 2*d_t)/(4*sigma**pi), -sigma**pi/4 - 1/(4*sigma**pi)],
[sigma**10*(d_t**2 + 2*d_t)/4 + (d_t**2 - 2*d_t)/(4*sigma**10),  sigma**10/4 + 1/(4*sigma**10)]])*Matrix([
[y_1(t)],
[y_2(t)]]))

## System analysis

We use the leading column coefficient matrices to get an overview.
As can be seen, the shifts in the second row of the first column are to big.

In [2]:
# convert to quasipolynomial matrix
coeff_dom = sp.EX.frac_field(d_sym)
qp_ring = ch.QuasiPolynomialRing(coeff_dom, s_sym)
gd_mat = sp.polys.matrices.DomainMatrix.from_list(g_mat.tolist(), domain=qp_ring)

print("Leading coefficient matrices:")
print("LCCM for derivatives:")
display(ch.get_lccm(gd_mat, d_sym).to_Matrix())
print("LCCM for shifts:")
display(ch.get_lccm(gd_mat, s_sym).to_Matrix())
print("LCCM for derivatives and shifts :")
lccm = ch.get_lccm(ch.get_lccm(gd_mat, d_sym), s_sym)
display(lccm)
print("Is G column-reduced w.r.t. shifts?")
ch.is_col_reduced(gd_mat, s_sym)

Leading coefficient matrices:
LCCM for derivatives:


Matrix([
[sigma**pi/4 + 1/(4*sigma**pi), -sigma**pi/4 - 1/(4*sigma**pi)],
[sigma**10/4 + 1/(4*sigma**10),  sigma**10/4 + 1/(4*sigma**10)]])

LCCM for shifts:


Matrix([
[               0,   0],
[d_t**2/4 + d_t/2, 1/4]])

LCCM for derivatives and shifts :


Matrix([
[  0,   0],
[1/4, 1/4]])

Is G column-reduced w.r.t. shifts?


False

## Shift reduction

We employ the developed algorithm to reduce the shifts


In [12]:
# Note: To keep the expressions short, the preliminary transformations are skipped

In [13]:
print("Computed transformation:")
gb_mat, l_mat = ch.reduce_matrix_shifts(gd_mat, d_sym, s_sym, check_cform=False)
display(l_mat.to_Matrix())

print("Transformed matrix:")
display(gb_mat.to_Matrix())

Computed transformation:


Matrix([
[                                                                                                                                                                                1, 0],
[-sigma**(-10 + pi) + sigma**(-10 + 3*pi)*(d_t/16 + 1/8)/(d_t/16 - 1/8) + sigma**(10 - 3*pi)*(d_t/16 - 1/8)/(d_t/16 + 1/8) - sigma**(10 - pi) - 16*exp(-20)*exp(6*pi)/(d_t**2 - 4), 1]])

Transformed matrix:


Matrix([
[                                                                                                                                                                              sigma**pi*(d_t**2/4 + d_t/2) + (d_t**2/4 - d_t/2)/sigma**pi,                                                                                                                                                                                                                                                                                             -sigma**pi/4 - 1/(4*sigma**pi)],
[-4*d_t*sigma**pi*exp(-20)*exp(6*pi)/(d_t - 2) - 4*d_t*exp(-20)*exp(6*pi)/(sigma**pi*(d_t + 2)) + sigma**(-10 + 4*pi)*(d_t**3/64 + d_t**2/16 + d_t/16)/(d_t/16 - 1/8) + sigma**(10 - 4*pi)*(d_t**3/64 - d_t**2/16 + d_t/16)/(d_t/16 + 1/8), sigma**10/2 + 4*sigma**pi*exp(-20)*exp(6*pi)/(d_t**2 - 4) - sigma**(-10 + 2*pi)/(64*(d_t/64 - 1/32)) + sigma**(-10 + 4*pi)*(-d_t/64 - 1/32)/(d_t/16 - 1/8) + sigma**(10 - 4*pi)*(1/32 - d_t/64)/(d_t/16 + 

In [14]:
print("Leading coefficient matrices:")
print("LCCM for derivatives:")
display(ch.get_lccm(gb_mat, d_sym).to_Matrix())

print("LCCM for shifts:")
display(ch.get_lccm(gb_mat, s_sym).to_Matrix())

print("LCCM for derivatives and shifts :")
lccm = ch.get_lccm(ch.get_lccm(gb_mat, d_sym), s_sym)
display(lccm)

print("Is the transformed G column-reduced w.r.t. derivatives and shifts?")
print(ch.is_col_reduced(gb_mat, s_sym))

print("How do the reduced shift intervals look like?")
funcs = y1, y2
shift_intervals = []
(lc, ld), (tc, td) = ch.get_matrix_terms(gb_mat, s_sym)
for idx, func in enumerate(funcs):
    shift_intervals.append((min(td[:,idx]), max(ld[:, idx])))
print(funcs)
print(shift_intervals)

print("How do the integrator chain lengths look like?")
derivative_orders = []
(lc, ld), _ = ch.get_matrix_terms(gb_mat, d_sym)
for idx, func in enumerate(funcs):
    derivative_orders.append(max(ld[:, idx]))
print(funcs)
print(derivative_orders)

Leading coefficient matrices:
LCCM for derivatives:


Matrix([
[               sigma**pi/4 + 1/(4*sigma**pi),                                               -sigma**pi/4 - 1/(4*sigma**pi)],
[sigma**(-10 + 4*pi)/4 + sigma**(10 - 4*pi)/4, sigma**10/2 - sigma**(-10 + 4*pi)/4 - sigma**(10 - 4*pi)/4 + 1/(2*sigma**10)]])

LCCM for shifts:


Matrix([
[                   d_t**2/4 + d_t/2,   0],
[-4*d_t*exp(-20)*exp(6*pi)/(d_t - 2), 1/2]])

LCCM for derivatives and shifts :


Matrix([
[1/4,   0],
[  0, 1/2]])

Is the transformed G column-reduced w.r.t. derivatives and shifts?
True
How do the reduced shift intervals look like?
(y_1(t), y_2(t))
[(-pi, pi), (-10, 10)]
How do the integrator chain lengths look like?
(y_1(t), y_2(t))
[2, 0]


## Stating the controller form

In this part we will:
* Apply the inverse Laplace transform and convolve input and output vector with the corresponding expressions.
* Simplify the resulting convolution integrals since they have with compact support
* Substitute expressions in $y$ that will not part of the new state
* Introduce controller form state and substitute into the FDE system


 #### Operations on the system input $u(t)$

As we applied an input transformation to reduce the shift degrees in the system,
 we now multiply the input with the inverse transformation $L^{-1}$.


In [15]:
print("Quasipolynomial Matrix:")
li_mat, _ = (l_mat * l0).inv_den()
display(li_mat.to_Matrix())

print("Convolution Kernel:")
li_conv_dmat = li_mat.applyfunc(lambda x: ch.inverse_laplace_transform(x, d_sym, s_sym, t))
li_conv_mat = li_conv_dmat.to_Matrix()
display(li_conv_mat)

print("Convolution:")
u_expr = sp.Integral(li_conv_mat.subs(t, tau) @ u_vec.subs(t, t + tau), (tau, -sp.oo, sp.oo)).doit()
display(u_expr)

print("Result after simplifying the expressions:")
u_expr = ch.simplify_conv_integrals(u_expr)
display(u_expr)

Quasipolynomial Matrix:


Matrix([
[                                                                                                                                                                                           sigma**(5 - pi/2),                  0],
[sigma**(-15 + 3*pi/2) + sigma**(-15 + 7*pi/2)*(-d_t/16 - 1/8)/(d_t/16 - 1/8) + 16*sigma**(-5 + pi/2)*exp(-20)*exp(6*pi)/(d_t**2 - 4) + sigma**(5 - 5*pi/2)*(1/8 - d_t/16)/(d_t/16 + 1/8) + sigma**(5 - pi/2), sigma**(-5 + pi/2)]])

Convolution Kernel:


Matrix([
[                                                                                                                                                                                                                                                                                                                                      DiracDelta(t - pi/2 + 5),                        0],
[4*exp(-2*t - 10 + 5*pi)*Heaviside(t - 5*pi/2 + 5) - 4*exp(-20)*exp(6*pi)*exp(-2*t - pi + 10)*Heaviside(t - 5 + pi/2) - 4*exp(2*t - 30 + 7*pi)*Heaviside(t - 15 + 7*pi/2) + 4*exp(-20)*exp(6*pi)*exp(2*t - 10 + pi)*Heaviside(t - 5 + pi/2) + DiracDelta(t - 15 + 3*pi/2) - DiracDelta(t - 15 + 7*pi/2) - DiracDelta(t - 5*pi/2 + 5) + DiracDelta(t - pi/2 + 5), DiracDelta(t - 5 + pi/2)]])

Convolution:


Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           u_1(t - 5 + pi/2)],
[exp(-20)*Integral(-4*u_1(t + tau)*exp(-10)*exp(7*pi)*exp(2*tau)*Heaviside(tau - 15 + 7*pi/2) + 4*u_1(t + tau)*exp(-10)*exp(7*pi)*exp(2*tau)*Heaviside(tau - 5 + pi/2) + u_1(t + tau)*exp(20)*DiracDelta(tau - 15 + 3*pi/2) - u_1(t + tau)*exp(20)*DiracDelta(tau - 15 + 7*pi/2) - u_1(t + tau)*exp(20)*DiracDelta(tau - 5*pi/2 + 5) + u_1(t + tau)*exp(20)*DiracDelta(tau - pi/2 + 5) - 4*u_1(t + tau)*exp(10)

Result after simplifying the expressions:


Matrix([
[                                                                                                                                                                                                                                                                                                                           u_1(t - 5 + pi/2)],
[(u_1(t - 5 + pi/2)*exp(20) - u_1(t - 5 + 5*pi/2)*exp(20) - u_1(t - 7*pi/2 + 15)*exp(20) + u_1(t - 3*pi/2 + 15)*exp(20) + u_2(t - pi/2 + 5)*exp(20) - 4*exp(10)*exp(5*pi)*Integral(u_1(t + tau)*exp(-2*tau), (tau, -5 + pi/2, 5 - 5*pi/2)) - 4*exp(-10)*exp(7*pi)*Integral(u_1(t + tau)*exp(2*tau), (tau, -15 + 7*pi/2, -5 + pi/2)))*exp(-20)]])

#### Operations on the flat output $y(t)$

Here, we used the transformed $G$ matrix as convolution kernel.

In [19]:
print("Quasipolynomial Matrix:")
display(gb_mat.to_Matrix())

print("Convolution Kernel:")
g_conv_dmat = gb_mat.applyfunc(lambda x: ch.inverse_laplace_transform(x, d_sym, s_sym, t))
g_conv_mat = g_conv_dmat.to_Matrix()
display(g_conv_mat)

print("Convolution:")
y_expr = sp.Integral(g_conv_mat.subs(t, tau) @ y_vec.subs(t, t + tau), (tau, -sp.oo, sp.oo)).doit()
display(y_expr)

print("Result after simplifying the expressions:")
y_expr = ch.simplify_conv_integrals(y_expr)
display(y_expr)

print("Result after elimination of non-state expressions:")
for i in range(len(funcs)):
    y_expr = ch.substitute_non_state_terms(y_expr, funcs[i], derivative_orders[i], *shift_intervals[i])
display(y_expr)

Quasipolynomial Matrix:


Matrix([
[                                                                                                                                                                              sigma**pi*(d_t**2/4 + d_t/2) + (d_t**2/4 - d_t/2)/sigma**pi,                                                                                                                                                                                                                                                                                             -sigma**pi/4 - 1/(4*sigma**pi)],
[-4*d_t*sigma**pi*exp(-20)*exp(6*pi)/(d_t - 2) - 4*d_t*exp(-20)*exp(6*pi)/(sigma**pi*(d_t + 2)) + sigma**(-10 + 4*pi)*(d_t**3/64 + d_t**2/16 + d_t/16)/(d_t/16 - 1/8) + sigma**(10 - 4*pi)*(d_t**3/64 - d_t**2/16 + d_t/16)/(d_t/16 + 1/8), sigma**10/2 + 4*sigma**pi*exp(-20)*exp(6*pi)/(d_t**2 - 4) - sigma**(-10 + 2*pi)/(64*(d_t/64 - 1/32)) + sigma**(-10 + 4*pi)*(-d_t/64 - 1/32)/(d_t/16 - 1/8) + sigma**(10 - 4*pi)*(1/32 - d_t/64)/(d_t/16 + 

Convolution Kernel:


Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           -DiracDelta(t - pi, 1)/2 + DiracDelta(t - pi, 2)/4 + DiracDelta(t + pi, 1)/2 + DiracDelta(t + pi, 2)/4,                                                                                                                                                                                                                                                                                                                                                                                                            

Convolution:


Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

Result after simplifying the expressions:


Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    -y_2(t - pi)/4 - y_2(t + pi)/4 - Subs(Derivative(y_1(_xi_1), _xi_1), _xi_1, t - pi)/2 + Subs(Derivative(y_1(_xi_1), _xi_1), _xi_1, t + pi)/2 + Subs(Derivative(y_1(_xi_1), (_xi_1, 2)), _xi_1, t - pi)/4 + Subs(Derivative(y_1(_xi_1), (_xi_1, 2)), _xi_1, t + pi)/4],
[(-16*y_1(t - pi)*exp(6*pi) - 16*y_

Result after elimination of non-state expressions:


Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              


### Solve for highest predictions of highest derivatives


In [20]:
# define terms to solve for
terms = []
for i in range(len(y_vec)):
    # TODO: find better work-around for dynamically introduced placeholder variables
    d = next(s for s in list(y_expr.atoms(sp.Symbol)) if "xi" in str(s))
    expr = y_vec[i].func(d).diff(d, derivative_orders[i]).subs(d, t + shift_intervals[i][1])
    terms.append(expr)

# solve io-form for expressions
io_form = y_expr - u_expr
sol = sp.solve(io_form, terms)

print("DDE-System:")
fde_form = sp.Matrix([sol[t] for t in terms])
fde_sys = sp.Eq(sp.Matrix(terms), fde_form)
display(fde_sys)

DDE-System:


Eq(Matrix([
[Subs(Derivative(y_1(_xi_1), (_xi_1, 2)), _xi_1, t + pi)],
[                                            y_2(t + 10)]]), Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

### Introduce the HCF state



In [22]:
eta_vec, y_eta_map, eta_y_map = ch.build_hcf_state(y_vec, derivative_orders, shift_intervals, t, tau)
print("HCF State:")
display(eta_vec)

HCF State:


Matrix([
[\eta^O_{({1}, {1})}(t)],
[\eta^O_{({1}, {2})}(t)],
[    \eta^P_{1}(tau, t)],
[    \eta^P_{2}(tau, t)]])

### Substitute new state variables

In [23]:
print("System in hyperbolic controller form:")
eta_expr = ch.substitute_hcf_state(fde_sys, y_eta_map, t, tau)
display(eta_expr)

System in hyperbolic controller form:


Eq(Matrix([
[\eta^P_{1}(2*pi, t)],
[  \eta^P_{2}(20, t)]]), Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  


This concludes the example.